In [1]:
import yfinance as yf
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Multiple stocks from different industries
tickers = [
    'AAPL', 'MSFT', 'GOOGL',  # Tech
    'TSLA',                    # Growth
    'JPM',                     # Finance
    'JNJ'                      # Healthcare
]

def build_features(df):
    df = df.copy()
    df.columns = df.columns.droplevel(1)
    df['Return']         = df['Close'].pct_change()
    df['MA_10']          = df['Close'].rolling(10).mean()
    df['MA_50']          = df['Close'].rolling(50).mean()
    df['Volatility']     = df['Return'].rolling(10).std()
    df['Momentum']       = df['Close'] - df['MA_10']
    df['MA_10_slope']    = df['MA_10'].pct_change()
    df['MA_50_slope']    = df['MA_50'].pct_change()
    df['Price_vs_MA10']  = df['Close'] / df['MA_10']
    df['Price_vs_MA50']  = df['Close'] / df['MA_50']
    df['Volume_change']  = df['Volume'].pct_change()
    df['High_Low_range'] = (df['High'] - df['Low']) / df['Close']
    df['Target']         = (df['Return'].shift(-1) > 0).astype(int)
    df.dropna(inplace=True)
    return df

# Download and combine all stocks
all_data = []

for ticker in tickers:
    print(f"Downloading {ticker}...")
    raw = yf.download(ticker, start="2020-01-01", end="2026-05-12", progress=False)
    featured = build_features(raw)
    all_data.append(featured)

# Combine all into one dataframe
combined = pd.concat(all_data)
print(f"\nTotal rows: {combined.shape[0]}")

# Features and target
features = [
    'Return', 'Volatility', 'Momentum',
    'MA_10_slope', 'MA_50_slope',
    'Price_vs_MA10', 'Price_vs_MA50',
    'Volume_change', 'High_Low_range'
]

X = combined[features]
y = combined['Target']

# Split and train
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

# Evaluate
predictions = model.predict(X_test)
print("\nAccuracy:", accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

# Save new model
joblib.dump(model, 'stock_model_1.pkl')
print("\nModel saved!")


Total rows: 9282

Accuracy: 0.5288099084544965
              precision    recall  f1-score   support

           0       0.51      0.37      0.43       892
           1       0.54      0.67      0.60       965

    accuracy                           0.53      1857
   macro avg       0.53      0.52      0.51      1857
weighted avg       0.53      0.53      0.52      1857


Model saved!
